# 04. 투자 기록: 왜 샀고, 어떻게 됐고, 언제 틀렸다고 볼 것인가

01~03이 **판단을 만드는** 단계라면, 여기는 **판단을 기록하고 나중에 채점하는** 단계다.

| 파일 | 내용 | GitHub |
|---|---|---|
| `journal/theses/<티커>.yaml` | 내 문장으로 쓴 논거, 근거가 된 토론, **틀리면 팔 조건**, 답하지 못한 질문, 복기 | 공개 |
| `journal/trades.csv` | 날짜, 티커, buy/sell, 수량, 단가, 수수료 | **로컬에만** (.gitignore) |
| `docs/portfolio.md` | 비중·수익률(%)·지수 대비 성과·팔 조건 상태·논거 | 공개 (이 노트북이 생성) |
| `outputs/portfolio_private.md` | 수량·평균단가·금액까지 전부 | 로컬에만 |

- **틀리면 팔 조건**은 01 노트북의 지표 코드(`PPE_TURN`, `INV_DAYS` …)로 쓰면 최신 재무로 자동 확인한다. 숫자로 못 재는 조건은 `text:`로 쓰고 직접 확인한다.
- 수익률은 **회차** 단위다. 전량 매도하면 회차가 끝나고, 다시 사면 새 회차가 시작된다. 회차 수익률 = (매도 대금 + 현재 평가액) ÷ 매수 원금 합계 − 1, 현지 통화, 수수료 포함. 한 회차 안에서 사고팔기를 반복하면 매수 원금이 계속 쌓여 수익률이 실제보다 낮게 나온다.
- **같은 기간 지수**는 같은 날 같은 돈으로 자국 지수를 샀다고 치고 계산한다 (추가 매수도 그날 지수를 추가로 산 것으로, 매도는 같은 비율만큼 판 것으로). 매수일의 지수 데이터가 없으면 비워 둔다.
- 액면분할은 매매 기록에 `split` 줄로 남긴다. Yahoo 분할 이력과 비교해서 빠진 분할은 ⚠️로 알려 준다.
- `journal/trades.csv`가 없으면 `journal/examples/`의 **예시 데이터**로 돌아간다.
- 이 노트북은 화면에 **공개용 정보만** 보여 준다. 노트북 출력이 git에 올라가도 금액은 드러나지 않는다.

## 0. 증권사에서 체결 내역 가져오기 (선택)

`.env`에 증권사 API 키가 있으면 한국투자증권·키움증권의 체결 내역을 `journal/trades.csv`로 가져온다. **조회 기능만** 호출하도록 코드에서 막혀 있다. 같은 기간을 다시 가져와도 중복되지 않고, 손으로 적은 줄은 그대로 남는다. 화면에는 건수만 나온다.

```
KIS_APP_KEY=...        KIS_APP_SECRET=...      KIS_ACCOUNT=12345678-01    KIS_ENV=real
KIWOOM_APP_KEY=...     KIWOOM_SECRET_KEY=...   KIWOOM_ENV=real
```

- 한국투자증권: 국내 + 해외(일본·홍콩·중국·미국). 대만은 지원하지 않는다.
- 키움증권: 국내만 (REST API의 해외주식은 미국뿐).
- 액면분할·무상증자·입고처럼 매매가 아닌 변동은 가져오지 않는다. 리포트에 ⚠️가 뜨면 `split` 줄을 직접 추가한다.

In [1]:
SYNC_FROM = None    # 예: "2024-01-01" 로 바꾸고 실행하면 그날부터 오늘까지 가져온다
if SYNC_FROM:
    from research import brokers
    print(brokers.sync(SYNC_FROM))

In [2]:
import sys, json, re
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from research import financials as fin, playbook

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams.update({"font.family": ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"],
                     "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6,
                     "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"   # 범주 색 고정 순서
from research import journal

BASE = "KRW"        # 비중 계산용 기준 통화
REFRESH = False     # True: 가격·재무를 오늘 새로 받기
o = journal.build(base=BASE, refresh=REFRESH)
print("예시 데이터 사용 중" if o["is_example"] else "실제 매매 기록 사용 중", "| 논거", len(o["theses"]), "개 | 회차", len(o["eval"]), "개")
print("공개 리포트:", o["public_path"].relative_to(ROOT))
print("비공개 리포트:", o["private_path"].relative_to(ROOT), "(git에 올라가지 않음, 금액 포함)")

예시 데이터 사용 중 | 논거 4 개 | 회차 3 개
공개 리포트: docs\portfolio_example.md
비공개 리포트: outputs\portfolio_private.md (git에 올라가지 않음, 금액 포함)


## 1. 공개 리포트

GitHub에 올라가는 내용 그대로다. ⚠️ **기록 점검**이 보이면 고칠 것: 논거 없이 산 종목, 상태가 매매 기록과 어긋나는 논거 파일.

In [3]:
display(Markdown(o["public_md"]))

# 투자 기록

*기준일 2026-11-26 · 비중은 KRW 환산 평가액 기준 · 수익률은 현지 통화, 수수료 포함 · 같은 기간 지수는 같은 날 같은 돈을 지수에 넣고 뺐을 때 · 수량·단가·금액은 공개하지 않습니다.*

> ⚠️ **예시 데이터입니다.** 실제 매매 기록(journal/trades.csv)과 논거 파일이 아직 없습니다.

틀리면 팔 조건: 🔴 조건에 걸림 · 🟢 아직 아님 · ⚪ 숫자로 못 재는 조건(직접 확인) · ⚫ 데이터 없음

## 보유 중

| 종목 | 비중 | 수익률 | 같은 기간 지수 | 지수 대비 | 보유일 | 팔 조건 |
|---|---:|---:|---:|---:|---:|---|
| SK hynix (`000660.KS`) | 82.7% | +258.5% | +55.5% KOSPI | +202.9%p | 542 | 🟢 🟢 ⚪ |
| SAKURA internet (`3778.T`) | 17.3% | -11.9% | -3.5% Nikkei 225 | -8.5%p | 190 | 🟢 🟢 ⚪ |

## 관심 종목 (논거만 있고 아직 사지 않음)

| 종목 | 기록 시작 | 팔 조건 |
|---|---|---|
| Tokyo Electron (`8035.T`) | 2026-10-01 | 🟢 |

## 정리한 회차

| 종목 | 기간 | 수익률 | 같은 기간 지수 | 지수 대비 |
|---|---|---:|---:|---:|
| TSMC (`2330.TW`) | 2025-03-10 → 2025-09-15 | +24.4% | +12.9% TAIEX | +11.5%p |

## 종목별 논거

### SK hynix (`000660.KS`) · 보유

*기록 시작 2025-06-02 · 플레이북 `semiconductor`*

> (예시) HBM 공급이 제한된 동안은 메모리 사이클 고점이 예전보다 길다. 재고가 쌓이기 시작하면 그 가정이 깨진다.

**틀리면 팔 조건**

- 🟢 재고일수(기말 재고/매출원가) > 150일 (재고일수가 사이클 하락기 수준으로 늘면 공급 제한 가정이 틀린 것) — 최신값 `INV_DAYS_FY2025` 재고일수(기말 재고/매출원가) FY2025(12): 136일
- 🟢 매출총이익률 < 35% — 최신값 `GM_FY2025` 매출총이익률 FY2025(12): 60.4%
- ⚪ 주요 AI 가속기 고객의 HBM 공급사 다변화 발표

**복기**

- 2026-04-01: (예시) 목표 비중을 넘어서 1/3 매도. 논거는 유지.

### TSMC (`2330.TW`) · 정리

*기록 시작 2025-03-10 · 플레이북 `semiconductor`*

> (예시) 첨단 공정 가격 인상이 해외 공장 원가 상승을 상쇄한다.

**틀리면 팔 조건**

- 🟢 매출총이익률 < 50% — 최신값 `GM_FY2025` 매출총이익률 FY2025(12): 59.9%

**복기**

- 2025-09-15: (예시) 밸류에이션이 사이클 평균을 크게 넘어 전량 매도. 논거가 틀려서가 아니라 가격 때문.

### SAKURA internet (`3778.T`) · 보유

*기록 시작 2026-05-20 · 플레이북 `neocloud`*

> (예시) 정부 지원을 받는 일본 유일 규모의 국산 GPU 클라우드. 이익 악화는 선행 투자의 감가상각 때문이고,
> 가동률이 올라오면 유형자산 회전율과 마진이 함께 회복된다.

토론 기록: [3778_debate_dryrun.md](examples/3778_debate_dryrun.md)

**틀리면 팔 조건**

- 🟢 유형자산 회전율(매출/순유형자산) < 0.5x (설비가 매출로 바뀌지 않으면 '가동률 회복' 가정이 틀린 것) — 최신값 `PPE_TURN_FY2026` 유형자산 회전율(매출/순유형자산) FY2026(03): 0.76x
- 🟢 순차입금/EBITDA > 4x (증설을 빚으로만 하면 사이클 한 번에 무너질 수 있음) — 최신값 `NET_DEBT_EBITDA_FY2026` 순차입금/EBITDA FY2026(03): 2.40x
- ⚪ 정부 클라우드 사업자 지정 취소 또는 경제산업성 보조금 축소

**아직 답하지 못한 질문**

- GPU 설비 가동률과 고객 계약 기간 (유가증권보고서, 실적발표 Q&A)

**복기**

- 2026-08-10: (예시) 1분기 매출은 예상대로 늘었지만 감가상각이 더 빨리 늘었다. 논거 유지, 확신도는 낮춤.

### Tokyo Electron (`8035.T`) · 관심

*기록 시작 2026-10-01 · 플레이북 `semiconductor`*

> (예시) 첨단 패키징 장비 수요가 전공정 둔화를 메울 수 있는지 지켜보는 중.

**틀리면 팔 조건**

- 🟢 설비투자/매출 > 15% — 최신값 `CAPEX_REV_FY2026` 설비투자/매출 FY2026(03): 9.0%

**아직 답하지 못한 질문**

- 중국 매출 비중과 수출 규제 영향


## 2. 틀리면 팔 조건만 모아 보기

🔴가 하나라도 있으면 그 종목의 논거를 다시 쓰거나, 팔거나, **왜 조건을 바꾸는지**를 복기에 남긴다. 조건을 조용히 바꾸는 것이 가장 나쁜 습관이다.

In [4]:
rows = [{"종목": t, "조건": k["criterion"], "상태": journal.ICON[k["status"]], "근거 id": k.get("evidence") or "-",
         "최신값": k.get("text") or "-"} for t, ks in o["kills"].items() for k in ks]
pd.DataFrame(rows)

,종목,조건,상태,근거 id,최신값
0,000660.KS,재고일수(기말 재고/매출원가) > 150일,🟢,INV_DAYS_FY2025,재고일수(기말 재고/매출원가) FY2025(12): 136일
1,000660.KS,매출총이익률 < 35%,🟢,GM_FY2025,매출총이익률 FY2025(12): 60.4%
2,000660.KS,주요 AI 가속기 고객의 HBM 공급사 다변화 발표,⚪,-,-
3,2330.TW,매출총이익률 < 50%,🟢,GM_FY2025,매출총이익률 FY2025(12): 59.9%
4,3778.T,유형자산 회전율(매출/순유형자산) < 0.5x,🟢,PPE_TURN_FY2026,유형자산 회전율(매출/순유형자산) FY2026(03): 0.76x
5,3778.T,순차입금/EBITDA > 4x,🟢,NET_DEBT_EBITDA_FY2026,순차입금/EBITDA FY2026(03): 2.40x
6,3778.T,정부 클라우드 사업자 지정 취소 또는 경제산업성 보조금 축소,⚪,-,-
7,8035.T,설비투자/매출 > 15%,🟢,CAPEX_REV_FY2026,설비투자/매출 FY2026(03): 9.0%


## 3. 새 논거 파일 만들기

02 토론을 마쳤으면 저장된 토론(`outputs/debates/*.json`)에서 바로 시작할 수 있다. **토론 후 내 관점**이 논거 초안이 되고, 심판이 남긴 **풀리지 않은 질문**이 답하지 못한 질문 목록이 된다. 만든 뒤에는 YAML을 직접 열어 팔 조건을 채운다.

```python
debates = sorted((ROOT / "outputs" / "debates").glob("3778.T_*.json"))
journal.new_thesis("3778.T", "SAKURA internet", playbook="neocloud", debate_json=debates[-1])
```

## 4. 매매 기록하는 법

`journal/trades.csv`를 만들고 한 줄에 한 건씩 적는다 (형식은 [journal/examples/trades.csv](../journal/examples/trades.csv)와 같음).

```
date,ticker,side,qty,price,fee
2026-05-20,3778.T,buy,100,3000,500
2026-09-30,3778.T,split,2,
```

- `split`: 액면분할. qty에 1주당 새 주식 수(2:1이면 2), price는 비움
- 같은 날짜의 줄은 적힌 순서대로 처리한다
- 형식 오류는 **몇 번째 줄인지만** 알려 준다 (수량·단가는 오류 메시지에도 안 나옴: 노트북 출력이 git에 올라가므로)

복기는 논거 파일의 `reviews:`에 날짜와 함께 남긴다. 사고팔 때, 실적 발표 후, 팔 조건이 🔴가 됐을 때.